# Tracking Through Occlusion

Fine-tune SAM 2.1 Tiny on MOSEv2 and compare against its original checkpoint. Run this notebook on the configured CUDA RunPod. Each code cell owns one responsibility. Main model execution and downloads are owner actions; this notebook is delivered unexecuted.

Read `docs/runpod.md` and `docs/design.md` first. The initial archives are approximately85 GB. The pilot measures feasibility and projects cost before the fit; around$30 is a planning target, not a promise. No automatic memory/spend cutoff or workload reduction is used.

## 1. Locate this self-contained project

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "src" / "sam2_adapter.py").is_file()
    and (p / "notebooks" / "video_tracking_occlusion.ipynb").is_file()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

Import the notebook-facing functions. No data or model is acquired by these imports.

In [ ]:
from src.config import ProjectConfig
from src.runs import start_run, resume_run
from src.environment import inspect_environment
from src.data import acquire_dataset, audit_duplicates, freeze_splits, inspect_data
from src.sam2_adapter import ensure_model_assets, inspect_model
from src.pilot import run_pilot
from src.train import fine_tune
from src.evaluate import evaluate_local, render_comparison_examples
from src.benchmark import benchmark_tracking
from src.submissions import export_official, import_submission_receipt
from src.demo import launch_demo
from src.reporting import plot_learning_curves, make_comparison_figures, write_report

## 2. Define the immutable experiment

Use dataclass replacement here before creating a run if deliberately changing a setting. The default protocol is the approved 1,000-update experiment. A resumed run uses its saved configuration and rejects incompatible source/environment changes.

In [ ]:
config = ProjectConfig()
config

Select a new run with `None`, or explicitly name a previous run. This operational field is excluded from experimental source identity; the experimental configuration is not.

In [ ]:
run_name = None

In [ ]:
run = resume_run(run_name, root=PROJECT_ROOT) if run_name else start_run(config, root=PROJECT_ROOT)
run.path

## 3. Inspect CUDA readiness

This cell stops local Mac/CPU Run All before the bulk download. Package/device metadata is distinct from the later numerical pilot.

In [ ]:
inspect_environment(run, require_experiment_device=True)

## 4. Acquire and freeze the data

The following cell downloads the author archives and verifies the published checksums. Partial transport downloads can resume; checksum mismatches never become accepted data.

In [ ]:
acquire_dataset(run)

Audit exact frames and sampled visual matches across the training pool and official validation. Public original-source identities are unavailable, so these are conservative duplicate groups.

In [ ]:
audit_duplicates(run)

Freeze whole-group partitions: targets300/40/60; actual counts may differ slightly. Official-validation overlap groups are excluded from the local pool.

In [ ]:
freeze_splits(run)

In [ ]:
inspect_data(run)

## 5. Acquire and inspect SAM 2.1 Tiny

Record the official checkpoint URL and downloaded content identity. The native model and trainable/frozen parameter lists are reconstructed explicitly.

In [ ]:
ensure_model_assets(run)

In [ ]:
inspect_model(run)

## 6. Run the GPU pilot

This performs real, discarded diagnostic updates: at least20 representative optimizer updates, a deliberate disappearance case, frozen/gradient checks, checkpoint recovery, a four-clip learning check and full-clip tracking. Read the returned forecast before executing the next fit. A failed gate is a failure to investigate, not permission to reduce the workload silently.

In [ ]:
pilot_result = run_pilot(run)
pilot_result

## 7. Record the pretrained development reference

Development is used for selection; local test and official validation are reserved for after selection.

In [ ]:
evaluate_local(run, checkpoint='initial', split='dev', protocol='point')

## 8. Fine-tune

1,000 optimizer updates, four accumulated clips per update, first-frame-only prompt mixture and no later training corrections. Every200 updates selects by development point-prompt J&F-dot; interrupted partial updates replay from the last completed checkpoint.

In [ ]:
fine_tune(run)

In [ ]:
plot_learning_curves(run)

## 9. Evaluate the frozen held-out protocols

Each checkpoint/protocol has its own cell. One-point results use the public first/last exclusion. For correction benefit, compare paired aggregates with the midpoint excluded in both arms. Ground-truth-derived corrective clicks are simulated annotation assistance.

In [ ]:
evaluate_local(run, checkpoint='initial', split='test', protocol='point')

In [ ]:
evaluate_local(run, checkpoint='best', split='test', protocol='point')

In [ ]:
evaluate_local(run, checkpoint='initial', split='test', protocol='point_correction')

In [ ]:
evaluate_local(run, checkpoint='best', split='test', protocol='point_correction')

In [ ]:
evaluate_local(run, checkpoint='initial', split='test', protocol='mask')

In [ ]:
evaluate_local(run, checkpoint='best', split='test', protocol='mask')

## 10. Benchmark tracking

Fixed development clips, one warm-up and three measured trajectories, 1/2/4 objects where available. Transfer, setup, CUDA-synchronized model and rendering costs have separate boundaries. This is not a cross-framework experiment.

In [ ]:
benchmark_tracking(run)

## 11. Inspect saved examples and figures

Selection is deterministic: greatest increase, greatest decrease and lowest adapted score. These illustrate failures as well as improvements. Playback FPS is a declared visualization choice, not an inferred original capture rate.

In [ ]:
render_comparison_examples(run, protocol='point')

In [ ]:
render_comparison_examples(run, protocol='point_correction')

In [ ]:
make_comparison_figures(run)

## 12. Export official validation

Each call performs complete official first-frame-mask inference for433 clips and produces a validated ZIP. No corrective prompts, account action, upload, or submission occurs here.

In [ ]:
export_official(run, checkpoint='initial')

In [ ]:
export_official(run, checkpoint='best')

## 13. Optional manual receipt import

Submit the packages yourself through the current MOSEv2 Codabench validation phase. Save the actual receipts. Fill only these operational inputs, then rerun the corresponding import/report cells. Values below are deliberately empty. Preserve server units in the transcribed metric names; do not use the returned scores to tune.

In [ ]:
initial_receipt_path = None
initial_submission_id = ''
initial_reported_metrics = {}

In [ ]:
initial_receipt = import_submission_receipt(run, 'initial', Path(initial_receipt_path), submission_id=initial_submission_id, reported_metrics=initial_reported_metrics) if initial_receipt_path else 'Pending owner submission and receipt'
initial_receipt

In [ ]:
best_receipt_path = None
best_submission_id = ''
best_reported_metrics = {}

In [ ]:
best_receipt = import_submission_receipt(run, 'best', Path(best_receipt_path), submission_id=best_submission_id, reported_metrics=best_reported_metrics) if best_receipt_path else 'Pending owner submission and receipt'
best_receipt

## 14. Write the evidence-derived technical report

Missing official receipts or measurements are labelled pending. Local custom-split scores remain separate from official results.

In [ ]:
write_report(run)

## 15. Launch the interactive demo

Upload a video, choose objects and add positive/negative points or boxes. Correct earlier frames and compare independent pretrained/adapted states. Export indexed masks, prompt history and silent H.264 videos. Default binding is loopback: access through a tunnel; external binding requires credentials in environment variables. Close the demo when finished.

In [ ]:
demo = launch_demo(run)